# Customer Segmentation & Rule-Based Churn-Risk Scoring
**Task 3 | Transparent heuristic model**

This notebook creates tenure-based customer segments, a simple explainable risk score, and practical retention recommendations. It is a rule-based prioritization exercise, not a trained machine-learning model or calibrated churn probability.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('../Task-01-Data-Quality-EDA/cleaned_customer_churn.csv')
print('Rows:', len(df))
display(df.head())


## 1. Create behavior-based tenure segments
- Early-tenure: 0–12 months
- Mid-tenure: 13–24 months
- Long-tenure: 25+ months


In [ ]:
def tenure_segment(months):
    if months <= 12:
        return 'Early-tenure'
    if months <= 24:
        return 'Mid-tenure'
    return 'Long-tenure'

df['TenureSegment'] = df['TenureMonths'].apply(tenure_segment)
display(df['TenureSegment'].value_counts().rename_axis('Segment').to_frame('Customers'))


## 2. Transparent risk-score rules
| Signal | Points |
|---|---:|
| Month-to-month contract | +3 |
| Tenure 0–6 months | +2 |
| Tenure 7–12 months | +1 |
| 4 or more support tickets | +2 |
| 2–3 support tickets | +1 |

Risk bands: High = 5–7; Medium = 3–4; Low = 0–2. Signals not listed receive zero points. This is a hand-built prioritization rule, not a statistically fitted model.


In [ ]:
def score_customer(row):
    score, reasons = 0, []
    if row['ContractType'] == 'Month-to-Month':
        score += 3; reasons.append('Month-to-month contract (+3)')
    if row['TenureMonths'] <= 6:
        score += 2; reasons.append('Tenure 6 months or less (+2)')
    elif row['TenureMonths'] <= 12:
        score += 1; reasons.append('Tenure 7–12 months (+1)')
    if row['SupportTickets'] >= 4:
        score += 2; reasons.append('4+ support tickets (+2)')
    elif row['SupportTickets'] >= 2:
        score += 1; reasons.append('2–3 support tickets (+1)')
    return pd.Series({'RiskScore': score, 'ScoreReasons': '; '.join(reasons) if reasons else 'No listed risk signals'})

df[['RiskScore', 'ScoreReasons']] = df.apply(score_customer, axis=1)
df['RiskLevel'] = pd.cut(df['RiskScore'], bins=[-1, 2, 4, 7], labels=['Low', 'Medium', 'High'])
display(df[['CustomerID','TenureSegment','RiskScore','RiskLevel','ScoreReasons']])


## 3. Recommendation logic
Recommendations are based on visible customer attributes and should be reviewed by a human before action.


In [ ]:
def recommend(row):
    actions = []
    if row['TenureSegment'] == 'Early-tenure':
        actions.append('Run an onboarding check-in and review product adoption.')
    elif row['TenureSegment'] == 'Mid-tenure':
        actions.append('Schedule a value review and ask about unmet needs.')
    else:
        actions.append('Use a loyalty or renewal touchpoint; recognize continued tenure.')
    if row['ContractType'] == 'Month-to-Month':
        actions.append('Offer a clear annual-plan comparison without pressure.')
    if row['SupportTickets'] >= 4:
        actions.append('Trigger proactive support follow-up and review repeat ticket themes.')
    elif row['SupportTickets'] >= 2:
        actions.append('Check whether open support issues were resolved.')
    return ' '.join(actions)

df['Recommendation'] = df.apply(recommend, axis=1)
output_cols = ['CustomerID','TenureMonths','ContractType','SupportTickets','TenureSegment','RiskScore','RiskLevel','ScoreReasons','Recommendation','Churn']
results = df[output_cols].copy()
results.to_csv('customer_segments_risk_recommendations.csv', index=False)
display(results)


## 4. Review segment and risk summaries


In [ ]:
display(df.groupby('TenureSegment', observed=False).agg(Customers=('CustomerID','count'), MeanTenure=('TenureMonths','mean'), Churned=('Churn', lambda s: (s == 'Yes').sum())))
display(df.groupby('RiskLevel', observed=False).agg(Customers=('CustomerID','count'), Churned=('Churn', lambda s: (s == 'Yes').sum())))
risk_counts = df['RiskLevel'].value_counts().reindex(['Low','Medium','High'], fill_value=0)
risk_counts.plot(kind='bar', title='Customers by Rule-Based Risk Level', ylabel='Customers', xlabel='Risk level')
plt.tight_layout(); plt.show()


## 5. Interpretation and limitations
- The sample creates 6 early-tenure, 5 mid-tenure and 4 long-tenure customers.
- The rules assign 6 High, 1 Medium and 8 Low risk customers.
- In this small sample, all 6 High and the single Medium customer are marked churned; none of the 8 Low customers are marked churned. This is an in-sample descriptive comparison, not evidence of predictive accuracy.
- The score has not been trained, calibrated, or validated on a separate test set. Do not interpret the score as a probability.
- Only 15 sample records are available. Larger historical data and business validation are needed before operational use.
